# BirdCLEF 2026 - Multi-Label EffNet/ConvNeXt Baseline
This notebook implements a sliding window multi-label classification pipeline for BirdCLEF 2026.

## 1. Setup & Imports

In [1]:
import os
import gc
import sys
import math
import time
import glob
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
import librosa
import soundfile as sf
from pathlib import Path

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler, ConcatDataset
import torchaudio
import torchaudio.transforms as T

import timm
import ast
import albumentations as A
from sklearn.metrics import average_precision_score, roc_auc_score # roc_auc_macro is the competition metric
from sklearn.model_selection import StratifiedKFold, train_test_split

import warnings
warnings.filterwarnings('ignore')

/usr/local/lib/python3.12/dist-packages/albumentations/check_version.py:147: UserWarning: Error fetching version info <urlopen error [Errno -3] Temporary failure in name resolution>
  data = fetch_version_info()


## 2. Configuration & Paths

In [2]:
class Config:
    ROOT_DIR = '/kaggle/input/competitions/birdclef-2026'
        
    TRAIN_CSV = os.path.join(ROOT_DIR, 'train.csv')
    TRAIN_AUDIO_DIR = os.path.join(ROOT_DIR, 'train_audio')
    SOUNDSCAPE_CSV = os.path.join(ROOT_DIR, 'train_soundscapes_labels.csv')
    SOUNDSCAPE_DIR = os.path.join(ROOT_DIR, 'train_soundscapes')

    MODEL_DIR = Path('/kaggle/input/models/timm/tf-efficientnet/pytorch/tf-efficientnet-b0/1/tf_efficientnet_b0_aa-827b6e33.pth')
    
    # Audio Setup
    SR = 32000
    WINDOW_SECONDS = 5
    HOP_SECONDS = 2.5  # For inference overlap
    
    # Mel Spectrogram Setup
    N_MELS = 128
    N_FFT = 2048
    HOP_LENGTH = 512
    FMIN = 20
    FMAX = 16000
    
    # Training Setup
    SEED = 42
    BATCH_SIZE = 32
    EPOCHS = 10
    LR = 1e-4
    WEIGHT_DECAY = 1e-4
    NUM_WORKERS = 0
    
    # Model Setup
    MODEL_NAME = 'tf_efficientnet_b0' 
    NUM_CLASSES = 0 
    
CFG = Config()

# Load the label sources up front.
print("Loading training and soundscape labels...")
train_df = pd.read_csv(CFG.TRAIN_CSV)
ss_df = pd.read_csv(CFG.SOUNDSCAPE_CSV)
sample_sub = pd.read_csv(os.path.join(CFG.ROOT_DIR, 'sample_submission.csv'))

# Keep the original clip labels for reporting, but train the head on the full
# competition label universe so the soundscape-only classes are not dropped.
train_labels = sorted(train_df['primary_label'].unique())
submission_labels = [c for c in sample_sub.columns if c != 'row_id']
unique_labels = submission_labels
label_to_id = {label: i for i, label in enumerate(unique_labels)}
id_to_label = {i: label for label, i in label_to_id.items()}

train_df['label_id'] = train_df['primary_label'].map(label_to_id)
CFG.NUM_CLASSES = len(unique_labels)

print(f"Detected {len(train_labels)} clip labels.")
print(f"Official submission labels: {CFG.NUM_CLASSES}")

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

Loading training and soundscape labels...
Detected 206 clip labels.
Official submission labels: 234
Using device: cuda


## 3. Utility Functions

In [3]:
def seed_everything(seed):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed_everything(CFG.SEED)

## 4. Dataset & Data Processing

In [4]:
class BirdDataset(Dataset):
    def __init__(self, df, audio_dir, transform=None, is_train=True):
        self.df = df
        self.audio_dir = audio_dir
        self.transform = transform
        self.is_train = is_train
        self.window_samples = CFG.SR * CFG.WINDOW_SECONDS
        
        self.mel_transform = T.MelSpectrogram(
            sample_rate=CFG.SR, n_fft=CFG.N_FFT, hop_length=CFG.HOP_LENGTH,
            n_mels=CFG.N_MELS, f_min=CFG.FMIN, f_max=CFG.FMAX
        )
        self.amplitude_to_db = T.AmplitudeToDB(top_db=80)
        
    def __len__(self):
        return len(self.df)
    
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        audio_path = os.path.join(self.audio_dir, row['filename'])
        
        try:
            # Speed Fix: sf.info + sf.read chunking
            info = sf.info(audio_path)
            total_samples = info.frames
            
            if total_samples > self.window_samples:
                if self.is_train:
                    start = random.randint(0, total_samples - self.window_samples)
                else:
                    start = 0
                y, _ = sf.read(audio_path, start=start, frames=self.window_samples, always_2d=True)
            else:
                y, _ = sf.read(audio_path, always_2d=True)
                
            y = y.mean(axis=1) # Mono
            
            if len(y) < self.window_samples:
                pad_len = self.window_samples - len(y)
                y = np.pad(y, (0, pad_len))
        except Exception as e:
            y = np.zeros(self.window_samples)
            
        # Background Noise Augmentation
        if self.is_train and random.random() < 0.5:
            noise = np.random.randn(len(y))
            y = y + 0.005 * noise
            
        y_tensor = torch.tensor(y, dtype=torch.float32)
        mel_spec = self.mel_transform(y_tensor)
        mel_spec = self.amplitude_to_db(mel_spec)
        mel_spec = (mel_spec - mel_spec.min()) / (mel_spec.max() - mel_spec.min() + 1e-6)
        image = torch.stack([mel_spec, mel_spec, mel_spec])
        
        target = torch.zeros(CFG.NUM_CLASSES, dtype=torch.float32)
        target[row['label_id']] = 1.0
        
        if 'secondary_labels' in row and pd.notna(row['secondary_labels']):
            
            try:
                sec_labels = ast.literal_eval(row['secondary_labels'])
                for sl in sec_labels:
                    if sl in label_to_id:
                        target[label_to_id[sl]] = 1.0
            except:
                pass
        
        return image, target

In [5]:
class SoundscapeDataset(Dataset):
    def __init__(self, df, audio_dir, transform=None):
        self.df = df
        self.audio_dir = audio_dir
        self.transform = transform
        self.window_samples = CFG.SR * CFG.WINDOW_SECONDS
        
        self.mel_transform = T.MelSpectrogram(
            sample_rate=CFG.SR, n_fft=CFG.N_FFT, hop_length=CFG.HOP_LENGTH,
            n_mels=CFG.N_MELS, f_min=CFG.FMIN, f_max=CFG.FMAX
        )
        self.amplitude_to_db = T.AmplitudeToDB(top_db=80)
        
    def __len__(self):
        return len(self.df)
    
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        audio_path = os.path.join(self.audio_dir, row['filename'])
        
        h, m, s = map(int, row['start'].split(':'))
        start_sample = (h * 3600 + m * 60 + s) * CFG.SR
        
        try:
            y, _ = sf.read(audio_path, start=start_sample, stop=start_sample + self.window_samples, always_2d=True)
            y = y.mean(axis=1) # Mono
            if len(y) < self.window_samples:
                y = np.pad(y, (0, self.window_samples - len(y)))
        except Exception as e:
            y = np.zeros(self.window_samples)
            
        y_tensor = torch.tensor(y, dtype=torch.float32)
        mel_spec = self.mel_transform(y_tensor)
        mel_spec = self.amplitude_to_db(mel_spec)
        mel_spec = (mel_spec - mel_spec.min()) / (mel_spec.max() - mel_spec.min() + 1e-6)
        image = torch.stack([mel_spec, mel_spec, mel_spec])
        
        target = torch.zeros(CFG.NUM_CLASSES, dtype=torch.float32)
        labels = str(row['primary_label']).split(';')
        for label in labels:
            if label in label_to_id:
                target[label_to_id[label]] = 1.0
                
        return image, target

## 5. Augmentations

In [6]:
# Utilities for SpecAugment, Mixup, etc.
def get_train_transforms():
    return A.Compose([
        A.CoarseDropout(max_holes=1, max_height=16, max_width=16, p=0.5),
    ])

def mixup_data(x, y, alpha=0.2):
    if alpha > 0:
        lam = np.random.beta(alpha, alpha)
    else:
        lam = 1
    batch_size = x.size()[0]
    index = torch.randperm(batch_size).cuda()
    mixed_x = lam * x + (1 - lam) * x[index, :]
    y_a, y_b = y, y[index]
    return mixed_x, y_a, y_b, lam

def mixup_criterion(criterion, pred, y_a, y_b, lam):
    return lam * criterion(pred, y_a) + (1 - lam) * criterion(pred, y_b)

## 6. Model Architecture

In [7]:
class BirdModel(nn.Module):
    def __init__(self, model_name, num_classes, model_path=None, pretrained=False):
        super().__init__()

        if model_path is not None:
            self.backbone = timm.create_model(model_name, checkpoint_path=model_path, pretrained=pretrained, in_chans=3)
        else:
            self.backbone = timm.create_model(model_name, pretrained=pretrained, in_chans=3)
        
        if 'efficientnet' in model_name:
            in_features = self.backbone.classifier.in_features
            self.backbone.classifier = nn.Identity()
        elif 'convnext' in model_name:
            in_features = self.backbone.head.fc.in_features
            self.backbone.head.fc = nn.Identity()
        else:
            in_features = self.backbone.get_classifier().in_features
            self.backbone.reset_classifier(0)
            
        self.head = nn.Linear(in_features, num_classes)
        
    def forward(self, x):
        features = self.backbone(x)
        out = self.head(features)
        return out

## 7. Loss & Optimization

In [8]:
def get_optimizer(model):
    return optim.AdamW(model.parameters(), lr=CFG.LR, weight_decay=CFG.WEIGHT_DECAY)

def get_scheduler(optimizer):
    return optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=CFG.EPOCHS, eta_min=1e-6)

def get_criterion():
    return nn.BCEWithLogitsLoss()  # Multi-label loss

## 8. Training Loops

In [9]:
def train_epoch(model, loader, optimizer, criterion, scaler, device):
    model.train()
    epoch_loss = 0
    for images, targets in tqdm(loader, desc='Train'):
        images, targets = images.to(device), targets.to(device)
        optimizer.zero_grad()
        
        # Mixup
        images, targets_a, targets_b, lam = mixup_data(images, targets)
        
        with torch.cuda.amp.autocast():
            outputs = model(images)
            loss = mixup_criterion(criterion, outputs, targets_a, targets_b, lam)
            
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        
        epoch_loss += loss.item()
        
    return epoch_loss / len(loader)

In [10]:
def valid_epoch(model, loader, criterion, device):
    model.eval()
    epoch_loss = 0
    preds, true_targets = [], []
    
    with torch.no_grad():
        for images, targets in tqdm(loader, desc='Valid'):
            images, targets = images.to(device), targets.to(device)
            outputs = model(images)
            loss = criterion(outputs, targets)
            epoch_loss += loss.item()
            
            preds.append(torch.sigmoid(outputs).cpu().numpy())
            true_targets.append(targets.cpu().numpy())
            
    preds = np.concatenate(preds)
    true_targets = np.concatenate(true_targets)
    
    # ROC-AUC skipping classes with no TPs
    auc_scores = []
    for i in range(CFG.NUM_CLASSES):
        # Only calculate AUC if there is at least one positive and one negative sample
        if len(np.unique(true_targets[:, i])) > 1:
            auc = roc_auc_score(true_targets[:, i], preds[:, i])
            auc_scores.append(auc)
            
    final_score = np.mean(auc_scores) if auc_scores else 0.0
    return epoch_loss / len(loader), final_score

## 9. Main Execution

In [11]:
# Load and Prepare Data
# train_df and ss_df are loaded in the config cell.
df = train_df.copy()

# Create a stratification label map over the full submission label space.
# The clip training set only covers a subset; the soundscape set fills the rest.
if 'label_id' not in df.columns:
    df['label_id'] = df['primary_label'].map(label_to_id)

# Duplicate rare classes so the stratified split stays valid.
counts = df['label_id'].value_counts()
rare_birds = counts[counts < 2].index.tolist()
if rare_birds:
    print(f"Found {len(rare_birds)} species with only 1 sample. Duplicating for stratification...")
    rare_df = df[df['label_id'].isin(rare_birds)].copy()
    df = pd.concat([df, rare_df], ignore_index=True)

# 80/20 split for Clips (unseen valid)
train_df, valid_df_clips = train_test_split(
    df, test_size=0.2, stratify=df['label_id'], random_state=CFG.SEED
)

# 80/20 split for Soundscapes (unseen valid) — FIX LEAKAGE
train_ss_df, valid_ss_df = train_test_split(
    ss_df, test_size=0.2, random_state=CFG.SEED
)

print(f"\n{'='*20} Zero-Leakage Split {'='*20}")
print(f"Train Clips: {len(train_df)} | Valid Clips: {len(valid_df_clips)}")
print(f"Train SS   : {len(train_ss_df)} | Valid SS   : {len(valid_ss_df)}")

Found 4 species with only 1 sample. Duplicating for stratification...

==================== Zero-Leakage Split ====================
Train Clips: 28442 | Valid Clips: 7111
Train SS   : 1182 | Valid SS   : 296


In [12]:
# Training Loop (Balanced Dataloaders & Disjoint OOF Validation)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# 'Reptilia' with only 1 sample
counts = df['label_id'].value_counts()
rare_birds = counts[counts < 2].index.tolist()

if rare_birds:
    print(f"Found {len(rare_birds)} species with only 1 sample. Duplicating for stratification...")
    rare_df = df[df['label_id'].isin(rare_birds)].copy()
    # Duplicate so each class has at least 2 samples for the stratified split
    df = pd.concat([df, rare_df], ignore_index=True)

In [13]:
# Splits already performed above (Clips + Soundscapes both 80/20).
print(f"Train Clips: {len(train_df)} | Valid Clips: {len(valid_df_clips)}")
print(f"Train SS   : {len(train_ss_df)} | Valid SS   : {len(valid_ss_df)}")

Train Clips: 28442 | Valid Clips: 7111
Train SS   : 1182 | Valid SS   : 296


In [14]:
# Weighted Samplers for BOTH (Training and Validation Balance)
def get_sampler(df_subset):
    class_counts = df_subset['label_id'].value_counts().sort_index().values
    class_weights = 1.0 / (class_counts + 1e-6)
    # Map weights to each sample
    weights = df_subset['label_id'].map(lambda x: class_weights[x] if x < len(class_weights) else 0).values
    return WeightedRandomSampler(weights=weights, num_samples=len(weights), replacement=True)

train_sampler = get_sampler(train_df)
valid_sampler = get_sampler(valid_df_clips)

In [15]:
# Datasets
train_clip_ds = BirdDataset(
    train_df,
    CFG.TRAIN_AUDIO_DIR,
    is_train=True
)

# Training Soundscapes: use 80% split only
train_soundscape_ds = SoundscapeDataset(
    train_ss_df,
    CFG.SOUNDSCAPE_DIR
)

train_ds = ConcatDataset([train_clip_ds, train_soundscape_ds])

# Validation Datasets: unseen 20% splits only
valid_ds_clips = BirdDataset(
    valid_df_clips,
    CFG.TRAIN_AUDIO_DIR,
    is_train=False
)

valid_ds_ss = SoundscapeDataset(
    valid_ss_df,  # <- Zero-leakage: unseen 20%
    CFG.SOUNDSCAPE_DIR
)

# Eval-only datasets for training set (no augmentation)
train_clip_eval_ds = BirdDataset(
    train_df,
    CFG.TRAIN_AUDIO_DIR,
    is_train=False  # No augmentation for eval
)
train_ss_eval_ds = SoundscapeDataset(
    train_ss_df,
    CFG.SOUNDSCAPE_DIR
)

# Loaders
# NOTE: sampler and shuffle are mutually exclusive in PyTorch.
# We pass sampler=train_sampler to activate class-balanced sampling.
train_loader = DataLoader(
    train_ds,
    batch_size=CFG.BATCH_SIZE,
    sampler=train_sampler,  # <-- Activated: replaces shuffle=True
    num_workers=CFG.NUM_WORKERS,
    pin_memory=True
)

# Separate eval loaders for training data (Clips & SS)
train_clip_eval_loader = DataLoader(
    train_clip_eval_ds,
    batch_size=CFG.BATCH_SIZE,
    shuffle=False,
    num_workers=CFG.NUM_WORKERS,
    pin_memory=True
)

train_ss_eval_loader = DataLoader(
    train_ss_eval_ds,
    batch_size=CFG.BATCH_SIZE,
    shuffle=False,
    num_workers=CFG.NUM_WORKERS,
    pin_memory=True
)

valid_loader_clips = DataLoader(
    valid_ds_clips,
    batch_size=CFG.BATCH_SIZE,
    shuffle=False,
    num_workers=CFG.NUM_WORKERS,
    pin_memory=True
)

valid_loader_ss = DataLoader(
    valid_ds_ss,
    batch_size=CFG.BATCH_SIZE,
    shuffle=False,
    num_workers=CFG.NUM_WORKERS,
    pin_memory=True
)

In [16]:
# Initialization
model = BirdModel(model_name=CFG.MODEL_NAME, model_path=CFG.MODEL_DIR, num_classes=CFG.NUM_CLASSES).to(device)
optimizer = get_optimizer(model)
scheduler = get_scheduler(optimizer)
criterion = get_criterion()
scaler = torch.cuda.amp.GradScaler()

best_score = 0
for epoch in range(1, CFG.EPOCHS+1):
    start_time = time.time()
    
    # Train
    train_loss = train_epoch(model, train_loader, optimizer, criterion, scaler, device)
    
    # --- 4-Way Pure Metric Evaluation ---
    # Train metrics: evaluated on TRAINING data only (no leakage)
    _, train_auc_clips = valid_epoch(model, train_clip_eval_loader, criterion, device)
    _, train_auc_ss   = valid_epoch(model, train_ss_eval_loader,   criterion, device)
    
    # Valid metrics: evaluated on UNSEEN data only (no leakage)
    _, val_auc_clips = valid_epoch(model, valid_loader_clips, criterion, device)
    _, val_auc_ss    = valid_epoch(model, valid_loader_ss,    criterion, device)
    
    scheduler.step()
    
    # --- Overfit Penalties (pure, apples-to-apples) ---
    p_clip = abs(train_auc_clips - val_auc_clips)  # Monitoring only
    p_ss   = abs(train_auc_ss   - val_auc_ss)      # Used in checkpoint metric
    
    # --- Robust Score ---
    # Primary objective = val_auc_ss (matches competition: test set is soundscapes)
    # Penalty = SS overfit gap only (apples-to-apples with objective)
    robust_score = val_auc_ss - p_ss
    
    # --- Per-Epoch Log ---
    duration = time.time() - start_time
    print(f"Epoch {epoch} | Loss: {train_loss:.4f} | Time: {int(duration)}s")
    print(f"  Clips -> Train: {train_auc_clips:.4f} | Val: {val_auc_clips:.4f} | Gap: {p_clip:.4f}  [monitoring]")
    print(f"  SS    -> Train: {train_auc_ss:.4f}   | Val: {val_auc_ss:.4f}   | Gap: {p_ss:.4f}  [checkpoint]")
    print(f"  Robust Score (val_ss - ss_gap): {robust_score:.4f}")
    
    if robust_score > best_score:
        best_score = robust_score
        torch.save(model.state_dict(), "best_model.pth")
        print(f"  !!! NEW BEST MODEL SAVED | Robust: {best_score:.4f} | Val SS: {val_auc_ss:.4f} | SS Overfit Gap: {p_ss:.4f} !!!")

Train:   0%|          | 0/889 [00:00<?, ?it/s]

Valid:   0%|          | 0/889 [00:00<?, ?it/s]

Valid:   0%|          | 0/37 [00:00<?, ?it/s]

Valid:   0%|          | 0/223 [00:00<?, ?it/s]

Valid:   0%|          | 0/10 [00:00<?, ?it/s]

...

Valid:   0%|          | 0/889 [00:00<?, ?it/s]

Valid:   0%|          | 0/37 [00:00<?, ?it/s]

Valid:   0%|          | 0/223 [00:00<?, ?it/s]

Valid:   0%|          | 0/10 [00:00<?, ?it/s]

Epoch 10 | Loss: 0.0128 | Time: 1058s
  Clips -> Train: 0.9112 | Val: 0.8889 | Gap: 0.0223  [monitoring]
  SS    -> Train: 0.5980   | Val: 0.6291   | Gap: 0.0312  [checkpoint]
  Robust Score (val_ss - ss_gap): 0.5980


## 10. Inference & Submission

In [17]:
# Load Best Model
print("Loading best model for inference...")
model = BirdModel(CFG.MODEL_NAME, CFG.NUM_CLASSES).to(device)
try:
    model.load_state_dict(torch.load('best_model.pth', map_location=device))
    model.eval()
    print("Model loaded successfully.")
except Exception as e:
    print(f"Warning: Could not load 'best_model.pth'. Make sure training completed. Error: {e}")

Loading best model for inference...
Model loaded successfully.


In [18]:
# Fallback Logic
TEST_DIR = os.path.join(CFG.ROOT_DIR, 'test_soundscapes')
test_files = []
if os.path.exists(TEST_DIR):
    test_files = sorted(glob.glob(f'{TEST_DIR}/*.ogg'))

if len(test_files) == 0:
    print('FALLBACK ACTIVE: No test files found. Using training soundscapes as dry-run.')
    test_files = sorted(glob.glob(f'{CFG.SOUNDSCAPE_DIR}/*.ogg'))[:5] # Use first 5 for speed
    IS_DRY_RUN = True
else:
    print(f'Found {len(test_files)} files in test directory.')
    IS_DRY_RUN = False

FALLBACK ACTIVE: No test files found. Using training soundscapes as dry-run.


In [19]:
# Inference Setup
mel_transform = T.MelSpectrogram(
    sample_rate=CFG.SR, n_fft=CFG.N_FFT, hop_length=CFG.HOP_LENGTH,
    n_mels=CFG.N_MELS, f_min=CFG.FMIN, f_max=CFG.FMAX
).to(device)
amplitude_to_db = T.AmplitudeToDB(top_db=80).to(device)

all_predictions = []
all_row_ids = []

In [20]:
# Sliding Window Inference Loop
print(f"\n{'='*20} Starting Inference {'='*20}")
for audio_path in tqdm(test_files, desc="Processing Soundscapes"):
    filename = os.path.basename(audio_path).replace('.ogg', '')
    
    try:
        # Load full audio
        y, _ = sf.read(audio_path, always_2d=True)
        y = y.mean(axis=1) # Mono
    except Exception as e:
        print(f"Error reading {audio_path}: {e}")
        continue
        
    y_tensor = torch.tensor(y, dtype=torch.float32).to(device)
    total_samples = len(y_tensor)
    window_samples = CFG.SR * CFG.WINDOW_SECONDS
    
    # Calculate number of segments
    n_segments = math.ceil(total_samples / window_samples)
    
    for seg_idx in range(n_segments):
        start_sample = seg_idx * window_samples
        end_sample = start_sample + window_samples
        end_time_sec = (seg_idx + 1) * CFG.WINDOW_SECONDS
        row_id = f"{filename}_{end_time_sec}"
        
        # Extract and pad segment if needed
        segment = y_tensor[start_sample:end_sample]
        if len(segment) < window_samples:
            segment = F.pad(segment, (0, window_samples - len(segment)))
            
        # Transform
        with torch.no_grad():
            mel_spec = mel_transform(segment)
            mel_spec = amplitude_to_db(mel_spec)
            mel_spec = (mel_spec - mel_spec.min()) / (mel_spec.max() - mel_spec.min() + 1e-6)
            image = torch.stack([mel_spec, mel_spec, mel_spec]).unsqueeze(0) # Add batch dimension
            
            # Predict
            output = model(image)
            probs = torch.sigmoid(output).squeeze(0).cpu().numpy()
            
        all_row_ids.append(row_id)
        all_predictions.append(probs)


==================== Starting Inference ====================


Processing Soundscapes:   0%|          | 0/5 [00:00<?, ?it/s]

In [21]:
# Submission Formatting
print("\nFormatting submission...")

# Keep the model output aligned to the training label set, then expand
# to the official competition schema before writing the CSV.
prediction_df = pd.DataFrame(all_predictions, columns=unique_labels)
submission_df = prediction_df.reindex(columns=submission_labels, fill_value=0.0)
submission_df.insert(0, 'row_id', all_row_ids)
submission_df = submission_df[['row_id'] + submission_labels]

expected_cols = len(submission_labels) + 1
if submission_df.shape[1] != expected_cols:
    raise ValueError(f"Submission has {submission_df.shape[1]} columns, expected {expected_cols}.")
if len(submission_df) != len(all_row_ids):
    raise ValueError(f"Submission has {len(submission_df)} rows, expected {len(all_row_ids)}.")
if submission_df.isnull().values.any():
    raise ValueError("Submission contains missing values.")

# Save to CSV
submission_path = 'submission.csv'
submission_df.to_csv(submission_path, index=False)

print(f"Submission saved to {submission_path}")
print(f"Shape: {submission_df.shape}")
print(f"First few rows:")
display(submission_df.head(3))


Formatting submission...
Submission saved to submission.csv
Shape: (60, 235)
First few rows:


,row_id,1161364,116570,1176823,1491113,1595929,209233,22930,22956,22961,...,whnjay1,whtdov,whwpic1,y00678,yebcar,yebela1,yecmac,yecpar,yehcar1,yeofly1
0,BC2026_Train_0001_S08_20250606_030007_5,0.016366,0.003688,0.000384,0.000141,0.002426,0.000151,0.004036,0.005687,0.001544,...,0.000042,0.003791,0.000494,0.001034,0.000206,0.001326,0.000165,0.002778,0.000090,0.000659
1,BC2026_Train_0001_S08_20250606_030007_10,0.012219,0.003298,0.000246,0.000124,0.005814,0.000126,0.002207,0.004371,0.001022,...,0.000044,0.003270,0.000599,0.001055,0.000185,0.001002,0.000148,0.002264,0.000097,0.000846
2,BC2026_Train_0001_S08_20250606_030007_15,0.015630,0.003387,0.000227,0.000074,0.003820,0.000088,0.002748,0.005226,0.001459,...,0.000016,0.002056,0.000297,0.000685,0.000106,0.000646,0.000078,0.001198,0.000045,0.000364
